# Sales Performance Analysis using SQL (SQLite)

## Project Objective
Analyze sales data to identify revenue trends, top customers, best-performing products, and category performance.

In [1]:
import sqlite3

# Create / connect to database
conn = sqlite3.connect("sales_project.db")
cursor = conn.cursor()

print("Database connected successfully!")

Database connected successfully!


In [2]:
# Create Customers table
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT
);
""")

# Create Products table
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price INTEGER
);
""")

# Create Orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    order_date DATE,
    quantity INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

conn.commit()

print("Tables created successfully!")

Tables created successfully!


In [3]:
# Insert customers
cursor.execute("DELETE FROM customers;")
cursor.executemany("""
INSERT INTO customers (customer_id, customer_name, city)
VALUES (?, ?, ?);
""", [
    (1, 'Aisha', 'Mumbai'),
    (2, 'Ravi', 'Delhi'),
    (3, 'Neha', 'Bangalore'),
    (4, 'Arjun', 'Chennai'),
    (5, 'Meera', 'Hyderabad')
])

# Insert products
cursor.execute("DELETE FROM products;")
cursor.executemany("""
INSERT INTO products (product_id, product_name, category, price)
VALUES (?, ?, ?, ?);
""", [
    (1, 'Laptop', 'Electronics', 60000),
    (2, 'Phone', 'Electronics', 30000),
    (3, 'Headphones', 'Accessories', 2000),
    (4, 'Shoes', 'Fashion', 4000),
    (5, 'Watch', 'Fashion', 7000)
])

# Insert orders
cursor.execute("DELETE FROM orders;")
cursor.executemany("""
INSERT INTO orders (order_id, customer_id, product_id, order_date, quantity)
VALUES (?, ?, ?, ?, ?);
""", [
    (1, 1, 1, '2024-01-10', 1),
    (2, 2, 2, '2024-01-15', 2),
    (3, 3, 3, '2024-02-05', 3),
    (4, 4, 4, '2024-02-20', 1),
    (5, 5, 5, '2024-03-01', 2),
    (6, 1, 2, '2024-03-10', 1),
    (7, 2, 3, '2024-04-12', 4),
    (8, 3, 1, '2024-04-18', 1),
    (9, 4, 5, '2024-05-05', 2),
    (10, 5, 4, '2024-05-15', 1)
])

conn.commit()

print("Data inserted successfully!")

Data inserted successfully!


In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sales_project.db")

In [5]:
pd.read_sql("SELECT * FROM customers;", conn)

,customer_id,customer_name,city
0,1,Aisha,Mumbai
1,2,Ravi,Delhi
2,3,Neha,Bangalore
3,4,Arjun,Chennai
4,5,Meera,Hyderabad


In [6]:
pd.read_sql("""
SELECT 
    o.order_id,
    c.customer_name,
    p.product_name,
    p.category,
    p.price,
    o.quantity,
    o.order_date,
    (p.price * o.quantity) AS revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
""", conn)

,order_id,customer_name,product_name,category,price,quantity,order_date,revenue
0,1,Aisha,Laptop,Electronics,60000,1,2024-01-10,60000
1,2,Ravi,Phone,Electronics,30000,2,2024-01-15,60000
2,3,Neha,Headphones,Accessories,2000,3,2024-02-05,6000
3,4,Arjun,Shoes,Fashion,4000,1,2024-02-20,4000
4,5,Meera,Watch,Fashion,7000,2,2024-03-01,14000
5,6,Aisha,Phone,Electronics,30000,1,2024-03-10,30000
6,7,Ravi,Headphones,Accessories,2000,4,2024-04-12,8000
7,8,Neha,Laptop,Electronics,60000,1,2024-04-18,60000
8,9,Arjun,Watch,Fashion,7000,2,2024-05-05,14000
9,10,Meera,Shoes,Fashion,4000,1,2024-05-15,4000


## 1.Total Revenue

In [7]:
pd.read_sql("""
SELECT SUM(p.price * o.quantity) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
""", conn)

,total_revenue
0,260000


## 2.Monthly Revenue Analysis

In [8]:
pd.read_sql("""
SELECT 
    strftime('%Y-%m', o.order_date) AS month, 
    SUM(p.price * o.quantity) AS monthly_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY month
ORDER BY month
""", conn)

,month,monthly_revenue
0,2024-01,120000
1,2024-02,10000
2,2024-03,44000
3,2024-04,68000
4,2024-05,18000


## 3.Top Customer

In [9]:
pd.read_sql("""
SELECT 
    c.customer_name,
    SUM(p.price * o.quantity) AS total_spent
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.customer_name
ORDER BY total_spent DESC
""", conn)

,customer_name,total_spent
0,Aisha,90000
1,Ravi,68000
2,Neha,66000
3,Meera,18000
4,Arjun,18000


## 4.Best-Selling Product

In [10]:
pd.read_sql("""
SELECT 
    p.product_name,
    SUM(p.price * o.quantity) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_revenue DESC
""", conn)

,product_name,total_revenue
0,Laptop,120000
1,Phone,90000
2,Watch,28000
3,Headphones,14000
4,Shoes,8000


## 5.Revenue by Category

In [11]:
pd.read_sql("""
SELECT 
    p.category,
    SUM(p.price * o.quantity) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
""", conn)

,category,total_revenue
0,Electronics,210000
1,Fashion,36000
2,Accessories,14000
